# Limpieza de Datos - Detección de Anomalías en Consumo Eléctrico

**Objetivo:** Preparar el dataset para un modelo de clasificación de anomalías (`Abnormal_Usage`).

**Acciones basadas en el EDA:**
- Eliminar columna `Meter_Id` (identificador sin valor predictivo)
- Convertir `Date` → columnas `Month` y `Year`
- Corregir tipos de `Expected_Energy(kwh)` y `Actual_Energy(kwh)` (object → float)
- Imputar valores faltantes con mediana
- Aplicar One-Hot Encoding a `Region_Code` y `Dwelling_Type`
- Estandarizar variables numéricas continuas (excl. discretas categóricas)
- Los outliers de `Usage_Deviation(%)` y `Temperature_C` se conservan de momento

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

print('Librerías cargadas correctamente')
print("Hola")

Librerías cargadas correctamente


## 2. Cargar el dataset

In [2]:
df_o = pd.read_csv('../data/Intelligent_abnormal_electricity_usage_dataset_REALWORLD.csv')

print(f'Dimensiones originales: {df_o.shape}')
df_o.head()

Dimensiones originales: (10800, 15)


,Meter_Id,Date,Region_Code,Dwelling_Type,Num_Occupants,House_Area (sqft),Appliance_Score,Connected_Load(kw),Temperature_C,Humidity (%),Expected_Energy(kwh),Actual_Energy(kwh),Usage_Deviation(%),Cluster_Avg_Energy(kwh),Abnormal_Usage
0,IN-KL-ELC-97400511,25-01-2023,IN_KL_TVM,Independent House,2,2458,6,7.21,29.52,83.53,12.82 kWh,19.886033739870022 kWh,55.12,16.89,1
1,IN-KL-ELC-28113115,04-01-2023,IN_KL_ALP,Independent House,3,2295,9,7.98,25.83,48.79,14.66 kWh,15.51 kWh,5.80,17.83,0
2,IN-KL-ELC-17499006,04-01-2023,IN_KL_TVM,Apartment,3,2424,16,5.47,31.91,46.66,16.67 kWh,15.11 kWh,-9.36,17.26,0
3,IN-KL-ELC-22187937,21-01-2023,IN_KL_ERN,Apartment,1,2787,18,5.41,21.15,58.49,12.71 kWh,21.26104008352499 kWh,67.28,16.61,1
4,IN-KL-ELC-57403818,28-01-2023,IN_KL_ERN,Independent House,4,2389,10,2.99,30.91,77.26,13.53 kWh,12.67 kWh,-6.36,16.43,0


## 3. Eliminar columnas no útiles

- `Meter_Id`: Es solo un identificador único, no aporta información predictiva.

In [3]:
df = df_o.copy()
df = df.drop(columns=['Meter_Id'])

print(f'Dimensiones después de eliminar Meter_Id: {df.shape}')
print(f'Columnas restantes: {df.columns.tolist()}')

Dimensiones después de eliminar Meter_Id: (10800, 14)
Columnas restantes: ['Date', 'Region_Code', 'Dwelling_Type', 'Num_Occupants', 'House_Area (sqft)', 'Appliance_Score', 'Connected_Load(kw)', 'Temperature_C', 'Humidity (%)', 'Expected_Energy(kwh)', 'Actual_Energy(kwh)', 'Usage_Deviation(%)', 'Cluster_Avg_Energy(kwh)', 'Abnormal_Usage']


## 4. Procesamiento de la columna `Date`

Se extraen `Month` y `Year`. No se incluye el día ya que no se considera relevante para el modelo.

In [4]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year

df = df.drop(columns=['Date'])

print('Nuevas columnas de fecha creadas:')
print(df[['Month', 'Year']].value_counts().sort_index())

Nuevas columnas de fecha creadas:
Month  Year
1      2023    10800
Name: count, dtype: int64


## 5. Corrección de tipos de datos en columnas de energía

`Expected_Energy(kwh)` y `Actual_Energy(kwh)` están como `object` porque contienen el sufijo `' kwh'` en el texto. Se elimina ese sufijo y se convierten a `float`.

In [5]:
# Ver ejemplos del formato actual
print('Ejemplos antes de limpiar:')
print(df['Expected_Energy(kwh)'].dropna().head(5).tolist())
print(df['Actual_Energy(kwh)'].dropna().head(5).tolist())

Ejemplos antes de limpiar:
['12.82 kWh', '14.66 kWh', '16.67 kWh', '12.71 kWh', '13.53 kWh']
['19.886033739870022 kWh', '15.51 kWh', '15.11 kWh', '21.26104008352499 kWh', '12.67 kWh']


In [6]:
def limpiar_kwh(valor):
    """Elimina el sufijo 'kwh' y convierte a float."""
    if pd.isna(valor):
        return np.nan
    return float((str(valor).lower()).replace(' kwh',''))

df['Expected_Energy(kwh)'] = df['Expected_Energy(kwh)'].apply(limpiar_kwh)
df['Actual_Energy(kwh)']   = df['Actual_Energy(kwh)'].apply(limpiar_kwh)

print('Tipos de datos después de la conversión:')
print(df[['Expected_Energy(kwh)', 'Actual_Energy(kwh)']].dtypes)
print('\nEjemplos después de limpiar:')
print(df['Expected_Energy(kwh)'].dropna().head(5).tolist())
print(df['Actual_Energy(kwh)'].dropna().head(5).tolist())

Tipos de datos después de la conversión:
Expected_Energy(kwh)    float64
Actual_Energy(kwh)      float64
dtype: object

Ejemplos después de limpiar:
[12.82, 14.66, 16.67, 12.71, 13.53]
[19.886033739870022, 15.51, 15.11, 21.26104008352499, 12.67]


## 6. Imputación de valores faltantes

Las columnas `Expected_Energy(kwh)` (~6.2%) y `Actual_Energy(kwh)` (~8.3%) tienen valores faltantes. Se imputa con la **mediana** para ser robusto frente a outliers.

In [7]:
# Verificar faltantes antes de imputar
print('Valores faltantes antes de imputar:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Valores faltantes antes de imputar:
Expected_Energy(kwh)    667
Actual_Energy(kwh)      900
dtype: int64


In [8]:
columnas_imputar = ['Expected_Energy(kwh)', 'Actual_Energy(kwh)']

for col in columnas_imputar:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f'{col} → mediana utilizada: {mediana:.4f}')

print('\nValores faltantes después de imputar:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('¡Sin valores faltantes!' if df.isnull().sum().sum() == 0 else '')

Expected_Energy(kwh) → mediana utilizada: 15.7500
Actual_Energy(kwh) → mediana utilizada: 17.2600

Valores faltantes después de imputar:
Series([], dtype: int64)
¡Sin valores faltantes!


## 7. Codificación de variables categóricas (One-Hot Encoding)

Se aplica One-Hot Encoding a `Region_Code` y `Dwelling_Type`. Se usa `drop_first=False` para conservar todas las categorías y facilitar la interpretación.

In [9]:
print('Categorías en Region_Code:', df['Region_Code'].unique())
print('Categorías en Dwelling_Type:', df['Dwelling_Type'].unique())

Categorías en Region_Code: ['IN_KL_TVM' 'IN_KL_ALP' 'IN_KL_ERN']
Categorías en Dwelling_Type: ['Independent House' 'Apartment' 'Villa']


In [10]:
df = pd.get_dummies(df, columns=['Region_Code', 'Dwelling_Type'], drop_first=False, dtype=int)

print(f'Dimensiones después del One-Hot Encoding: {df.shape}')
print('\nNuevas columnas generadas:')
nuevas_cols = [c for c in df.columns if c.startswith('Region_Code') or c.startswith('Dwelling_Type')]
print(nuevas_cols)

Dimensiones después del One-Hot Encoding: (10800, 19)

Nuevas columnas generadas:
['Region_Code_IN_KL_ALP', 'Region_Code_IN_KL_ERN', 'Region_Code_IN_KL_TVM', 'Dwelling_Type_Apartment', 'Dwelling_Type_Independent House', 'Dwelling_Type_Villa']


## 8. Estandarización de variables numéricas continuas

Se estandarizan las variables **continuas**. Se excluyen:
- `Num_Occupants` y `Appliance_Score`: variables discretas de carácter categórico ordinal.
- `Abnormal_Usage`: variable objetivo.
- Las columnas OHE y las de fecha (`Month`, `Year`).

In [11]:
# Variables a estandarizar (continuas)
cols_no_estandarizar = [
    'Num_Occupants',
    'Appliance_Score',
    'Abnormal_Usage',
    'Month',
    'Year'
]

# Excluir también las columnas OHE y la variable objetivo
ohe_cols = [c for c in df.columns if c.startswith('Region_Code') or c.startswith('Dwelling_Type')]
cols_no_estandarizar += ohe_cols

cols_estandarizar = [c for c in df.select_dtypes(include=[np.number]).columns
                     if c not in cols_no_estandarizar]

print('Columnas que se estandarizarán:')
print(cols_estandarizar)

Columnas que se estandarizarán:
['House_Area (sqft)', 'Connected_Load(kw)', 'Temperature_C', 'Humidity (%)', 'Expected_Energy(kwh)', 'Actual_Energy(kwh)', 'Usage_Deviation(%)', 'Cluster_Avg_Energy(kwh)']


In [12]:
scaler = StandardScaler()
df[cols_estandarizar] = scaler.fit_transform(df[cols_estandarizar])

print('Estadísticas después de estandarizar (media ≈ 0, std ≈ 1):')
df[cols_estandarizar].describe().loc[['mean', 'std']].round(4)

Estadísticas después de estandarizar (media ≈ 0, std ≈ 1):


,House_Area (sqft),Connected_Load(kw),Temperature_C,Humidity (%),Expected_Energy(kwh),Actual_Energy(kwh),Usage_Deviation(%),Cluster_Avg_Energy(kwh)
mean,0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 9. Verificación final del dataset limpio

In [13]:
print('=== RESUMEN DEL DATASET LIMPIO ===')
print(f'Dimensiones: {df.shape}')
print(f'\nValores faltantes totales: {df.isnull().sum().sum()}')
print(f'\nTipos de datos:')
print(df.dtypes)
print(f'\nDistribución de la variable objetivo:')
print(df['Abnormal_Usage'].value_counts())
print(f'\nBalance: Normal={df["Abnormal_Usage"].value_counts()[0]/len(df)*100:.1f}%  Anómalo={df["Abnormal_Usage"].value_counts()[1]/len(df)*100:.1f}%')

=== RESUMEN DEL DATASET LIMPIO ===
Dimensiones: (10800, 19)

Valores faltantes totales: 0

Tipos de datos:
Num_Occupants                        int64
House_Area (sqft)                  float64
Appliance_Score                      int64
Connected_Load(kw)                 float64
Temperature_C                      float64
Humidity (%)                       float64
Expected_Energy(kwh)               float64
Actual_Energy(kwh)                 float64
Usage_Deviation(%)                 float64
Cluster_Avg_Energy(kwh)            float64
Abnormal_Usage                       int64
Month                                int32
Year                                 int32
Region_Code_IN_KL_ALP                int64
Region_Code_IN_KL_ERN                int64
Region_Code_IN_KL_TVM                int64
Dwelling_Type_Apartment              int64
Dwelling_Type_Independent House      int64
Dwelling_Type_Villa                  int64
dtype: object

Distribución de la variable objetivo:
Abnormal_Usage
0    606

In [14]:
df.head()

,Num_Occupants,House_Area (sqft),Appliance_Score,Connected_Load(kw),Temperature_C,Humidity (%),Expected_Energy(kwh),Actual_Energy(kwh),Usage_Deviation(%),Cluster_Avg_Energy(kwh),Abnormal_Usage,Month,Year,Region_Code_IN_KL_ALP,Region_Code_IN_KL_ERN,Region_Code_IN_KL_TVM,Dwelling_Type_Apartment,Dwelling_Type_Independent House,Dwelling_Type_Villa
0,2,1.108194,6,1.225870,-0.108795,1.633974,-1.036012,0.319133,1.460891,0.228746,1,1,2023,0,0,1,0,1,0
1,3,0.878010,9,1.723842,-1.033049,-1.389365,-0.395372,-0.516777,-0.371645,2.110315,0,1,2023,1,0,0,0,1,0
2,3,1.060180,16,0.100582,0.489842,-1.574734,0.304458,-0.593185,-0.934931,0.969364,0,1,2023,0,0,1,1,0,0
3,1,1.572799,18,0.061779,-2.205274,-0.545198,-1.074312,0.581787,1.912708,-0.331721,1,1,2023,0,1,0,1,0,0
4,4,1.010754,10,-1.503276,0.239366,1.088311,-0.788809,-1.059273,-0.823463,-0.692021,0,1,2023,0,1,0,0,1,0


In [15]:
df.to_csv('../data/datos_limpios_anomalias.csv', index=False)
print("✓ Dataset limpio guardado en: ../data/datos_limpios_anomalias.csv")

# Verificar que se guardó correctamente
print(f"\nDimensiones del dataset guardado: {df.shape}")
print(f"Columnas guardadas: {list(df.columns)}")

✓ Dataset limpio guardado en: ../data/datos_limpios_anomalias.csv

Dimensiones del dataset guardado: (10800, 19)
Columnas guardadas: ['Num_Occupants', 'House_Area (sqft)', 'Appliance_Score', 'Connected_Load(kw)', 'Temperature_C', 'Humidity (%)', 'Expected_Energy(kwh)', 'Actual_Energy(kwh)', 'Usage_Deviation(%)', 'Cluster_Avg_Energy(kwh)', 'Abnormal_Usage', 'Month', 'Year', 'Region_Code_IN_KL_ALP', 'Region_Code_IN_KL_ERN', 'Region_Code_IN_KL_TVM', 'Dwelling_Type_Apartment', 'Dwelling_Type_Independent House', 'Dwelling_Type_Villa']



## Resumen de transformaciones realizadas

| Paso | Acción | Justificación |
|------|--------|---------------|
| 1 | Eliminar `Meter_Id` | Identificador único sin valor predictivo |
| 2 | `Date` → `Month`, `Year` | Captura estacionalidad; el día no es relevante |
| 3 | Limpiar sufijo `kwh` y convertir a `float` | Columnas de energía estaban como `object` |
| 4 | Imputar con mediana | ~6-8% faltantes en columnas de energía; mediana es robusta a outliers |
| 5 | OHE a `Region_Code` y `Dwelling_Type` | Variables categóricas nominales |
| 6 | StandardScaler en continuas | Mejora convergencia de modelos de ML |
| — | Outliers conservados | `Usage_Deviation(%)` y `Temperature_C` se dejan para evaluar impacto en el modelo |
